In [32]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import umap
from sklearn.datasets import load_digits


import hdbscan
import sklearn.cluster as cluster
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


In [33]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"

last_file="anonymous_answer_2026-07-24.csv"


In [57]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "r") as file_in:
    df00 = pd.read_csv(file_in, sep =",")

#list_affil = pd.read_csv("../list_affiliation.csv", sep =",")
df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2","H5X6-KL2Z",'PS8W-TJ9P'])].drop_duplicates(subset="q45_clé")#.merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")
df0_bis = df00.loc[df00.q45_clé.isin(["H5X6-KL2Z",'PS8W-TJ9P'])].drop_duplicates(subset="q45_clé")

In [58]:
df0_bis

,qno_obs,q1_so_principles,q2_hal_depot,q3_octavi_depot,q4_diff_data,q4_autres_entrepots,q5_r,q5_python,q5_excel,q5_julia,...,q25_doctorant,q3_octavi_rec,q5_1_other_lang_rec,q5_logiciel_traitement_rec,q5_family_logi,q6_1_dev_what,q6_1_dev_group,q6_1_dev_how,q26_1_autre_rec,q35_ia_tools_rec
0,1,"Oui, un peu","Oui, une fois",Non,Non,NaN,Non,Non,Non,Non,...,True,Non,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,58,"Oui, un peu",Non,Non,Non,NaN,Non,Non,Oui,Non,...,True,Non,NaN,excel,feuille de calcul,NaN,NaN,NaN,NaN,chatgpt|gemini|claude


In [59]:
df_clus = pd.read_csv("research_field_ind_VF.csv", sep=",")
df0["no_cluster"]= df0.q45_clé.map(dict(zip(df_clus.q45_clé, df_clus.no_cluster)))
df0["nom_cluster"]= df0.q45_clé.map(dict(zip(df_clus.q45_clé, df_clus.nom_cluster)))

In [60]:
df_clus.nom_cluster.unique()

array(['Sciences informatiques', 'Psychologie',
       'Individus, normes, sociétés', 'Art, cultures et communication',
       'Sciences historiques', 'Information et communication',
       'Recherche et création', 'Géographie'], dtype=object)

In [61]:
df0[["q45_clé","no_cluster"]].loc[df0.no_cluster.isna()]

,q45_clé,no_cluster
119,32TW-JLY5,NaN
120,6GVE-N5NJ,NaN


# Profil disciplinaire

In [62]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df0.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data, df_explode

In [63]:
df0["q24_research_fields"]= df0.apply(lambda row: row.q24_research_fields.replace(";","|") ,1 )


In [64]:
gb_data, df_exp = split_multiple_choices(df0, column='q24_research_fields', index = "q45_clé")


In [65]:
df_rf = df_exp[["q45_clé", 'q24_research_fields','no_cluster','nom_cluster']].copy()
df_rf.loc[df_rf.q24_research_fields.str.contains("SH"), "q24_research_domain"] = "Sciences sociales et humaines"
df_rf.loc[df_rf.q24_research_fields.str.contains("LS"), "q24_research_domain"] = "Sciences de la vie"
df_rf.loc[df_rf.q24_research_fields.str.contains("PE"), "q24_research_domain"] = "Sciences physiques et ingénierie"
df_rf["value"] = 1
df_rf.q24_research_fields.value_counts()
df_rf

,q45_clé,q24_research_fields,no_cluster,nom_cluster,q24_research_domain,value
1,GNEG-5QXQ,SH3 Le monde social et ses interactions,7.0,"Art, cultures et communication",Sciences sociales et humaines,1
1,GNEG-5QXQ,SH4 L'esprit humain et sa complexité,7.0,"Art, cultures et communication",Sciences sociales et humaines,1
1,GNEG-5QXQ,SH5 Textes et concepts,7.0,"Art, cultures et communication",Sciences sociales et humaines,1
1,GNEG-5QXQ,SH6 L'étude du passé humain,7.0,"Art, cultures et communication",Sciences sociales et humaines,1
2,FCCB-GMYA,SH4 L'esprit humain et sa complexité,4.0,Psychologie,Sciences sociales et humaines,1
...,...,...,...,...,...,...
119,32TW-JLY5,PE8 Ingénierie des produits et des procédés,NaN,NaN,Sciences physiques et ingénierie,1
119,32TW-JLY5,PE11 Génie des matériaux,NaN,NaN,Sciences physiques et ingénierie,1
120,6GVE-N5NJ,SH5 Textes et concepts,NaN,NaN,Sciences sociales et humaines,1
120,6GVE-N5NJ,SH6 L'étude du passé humain,NaN,NaN,Sciences sociales et humaines,1


In [66]:
df_rfi = df_rf.pivot(index =['q45_clé'], columns= 'q24_research_fields', values = "value").fillna(0).reset_index()
m_row = df_rfi[df_rfi.columns[1:]].values
df_rfi["no_cluster"] = df_rfi.q45_clé.map(dict(zip(df_clus.q45_clé, df_clus.no_cluster)))
#df_rfi["nom_cluster"] = df_rfi.no_cluster.map(dict(zip(df_clus.q45_clé, df_clus.nom_cluster)))

# Clusterisation des individus

Après le regroupement des champs disciplinaires, nous allons utilisés les mêmes algorithmes pour, cette fois, identifier les individus aux profils disciplinaires proches. L'exploration des données a mis en évidence l'existence de chercheurs qui "cochent" toutes les disciplines ou qui ont répondu deux fois (sans sélectionner le même nombre de disciplines). Nous commençons par enlevés ces lignes qui créent du bruit.

Ensuite, nous avons commencé par une approche "non-supervisée", puis au fur et à mesure de la classification des individus, nous avons utilisés les labels pour contruire des modèles semi-supervisé. Au final, nous avons identifiés neuf aires disciplinaires. Ces informations sont contenues dans les fichiers "tableau_cluster_ind.txt" et "tableau_cluster.txt". Le premier groupe que nous avons identifié



In [67]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [68]:
sns.set(context='notebook', style='white', rc={'figure.figsize':(14,10)})

In [69]:
df_lbl = df_rfi.loc[~df_rfi.no_cluster.isna()]
df_unlbl = df_rfi.loc[df_rfi.no_cluster.isna()].drop(columns=['q45_clé','no_cluster'])
df_unlbl

# 1. SÉPARATION DES DONNÉES D'ORIGINE
# On repart de vos DataFrames bruts (sans get_dummies)
# Séparer les fonctionnalités (X) et la cible (y)
X_labeled = df_lbl.drop(columns=['q45_clé', 'no_cluster'])
y = df_lbl['no_cluster']

# 2. ENCODAGE ORDINAL (Adapté pour le Naïve Bayes)
# handle_unknown='use_encoded_value' est CRUCIAL ici : si vos 2 nouvelles lignes
# contiennent une catégorie jamais vue, il lui donne un index spécial (-1) au lieu de planter.
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# On ajuste l'encodeur sur les données étiquetées et on transforme
X_encoded = encoder.fit_transform(X_labeled)
X_new_encoded = encoder.transform(df_unlbl)

# En Naïve Bayes, les valeurs négatives ne sont pas autorisées. 
# Si des catégories inconnues (-1) apparaissent dans les 2 lignes, on les ramène à 0.
X_new_encoded[X_new_encoded < 0] = 0

# 3. INITIALISATION DU MODÈLE NAÏVE BAYES
# min_categories force le modèle à comprendre la taille de votre dictionnaire de catégories
classifier_nb = CategoricalNB(min_categories=[len(cat) for cat in encoder.categories_])

# 4. VALIDATION CROISÉE
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_nb = cross_val_score(classifier_nb, X_encoded, y, cv=cv, scoring='f1_macro')

print("=== ÉVALUATION NAÏVE BAYES ===")
print(f"Moyenne globale du score F1 : {scores_nb.mean():.2f}")
print("==============================\n")

# 5. ENTRAÎNEMENT ET PRÉDICTION FINAL
classifier_nb.fit(X_encoded, y)
predictions_nb = classifier_nb.predict(X_new_encoded)

print(f"Prédictions Naïve Bayes pour vos 2 lignes : {predictions_nb}")


=== ÉVALUATION NAÏVE BAYES ===
Moyenne globale du score F1 : 0.84

Prédictions Naïve Bayes pour vos 2 lignes : [3. 7.]


In [70]:

# 9. Sauvegarde
df_unlbl['no_cluster'] = predictions_nb
df_unlbl["nom_cluster"] = df_unlbl.no_cluster.map(dict(zip(df_clus.no_cluster, df_clus.nom_cluster)))
df_unlbl1 = df_rfi.loc[df_rfi.no_cluster.isna()].drop(columns=['no_cluster']).reset_index()
df_unlbl = df_unlbl.reset_index()
df_unlbl["q45_clé"]= df_unlbl.index.map(dict(zip(df_unlbl1.index, df_unlbl1.q45_clé)))


In [71]:
df_lbl["nom_cluster"]= df_lbl.q45_clé.map(dict(zip(df_clus.q45_clé, df_clus.nom_cluster)))

/tmp/ipykernel_4967/474031760.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lbl["nom_cluster"]= df_lbl.q45_clé.map(dict(zip(df_clus.q45_clé, df_clus.nom_cluster)))


In [72]:
df_lbl = pd.concat([df_lbl, df_unlbl.drop(columns=["index"])])
df01 = df0.merge(df_lbl[["q45_clé","no_cluster","nom_cluster"]], on = ["q45_clé"], how="left")

In [73]:
df02 = pd.concat([df01, df0_bis])

In [77]:
# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/{last_file}", "w") as file_out:
    df02.to_csv(file_out, sep=",", index= False)

In [78]:
df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
149,16. recodage dev_exemple,q6_1_dev_what,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Recodage de la question 16. Restitue le type d...
150,16. recodage dev_exemple groupe,q6_1_dev_group,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Classe les exemples de logiciels développés da...
151,16. recodage dev_exemple how,q6_1_dev_how,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Précise le langage ou l'outil de programmation...
152,"113. recodage Autre, précisez",q26_1_autre_rec,13_env_travail,False,nominal_multiple,False,object,113.0,"Où travaillez-vous le plus souvent? Si autre, ...",q26,recode les autre lieux


In [79]:
dic_new_var = {'name':'Research area id',
               'label':'no_cluster',
               'group':'champ de recherche',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.no_cluster.dtypes,
               'no_question':135,
               'question':"",
               'question_family':'',
               'comment':"recode des champs de recherche"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
150,16. recodage dev_exemple groupe,q6_1_dev_group,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Classe les exemples de logiciels développés da...
151,16. recodage dev_exemple how,q6_1_dev_how,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Précise le langage ou l'outil de programmation...
152,"113. recodage Autre, précisez",q26_1_autre_rec,13_env_travail,False,nominal_multiple,False,object,113.0,"Où travaillez-vous le plus souvent? Si autre, ...",q26,recode les autre lieux
153,135. recodage ia_tools,q35_ia_tools,4_logiciel_traitement,False,nominal_multiple,False,object,135.0,"Si oui, lesquels ?",q35,recode les outils ia


In [80]:
dic_new_var = {'name':'Research area nom',
               'label':'nom_cluster',
               'group':'champ de recherche',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.nom_cluster.dtypes,
               'no_question':135,
               'question':"",
               'question_family':'',
               'comment':"recode des champs de recherche"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col2 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col2

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
151,16. recodage dev_exemple how,q6_1_dev_how,5_developpement,False,simple_nominal,False,object,16.0,"6.1/ Si oui, pourriez-vous expliquer ce que vo...",NaN,Précise le langage ou l'outil de programmation...
152,"113. recodage Autre, précisez",q26_1_autre_rec,13_env_travail,False,nominal_multiple,False,object,113.0,"Où travaillez-vous le plus souvent? Si autre, ...",q26,recode les autre lieux
153,135. recodage ia_tools,q35_ia_tools,4_logiciel_traitement,False,nominal_multiple,False,object,135.0,"Si oui, lesquels ?",q35,recode les outils ia
154,Research area id,no_cluster,champ de recherche,False,simple_nominal,False,float64,135.0,,,recode des champs de recherche


In [81]:
df_col2.to_csv("../le_questionnaire/dico_variable.csv", sep=",", index= False)